# 01 · Data processing and cleaning

**STA 556 Final Project** · Benjamin Owusu Acquah

This notebook turns the raw CDC PLACES data into the analysis-ready file used by `02_analysis.ipynb`. It documents every step: where the data come from, what the raw data look like, the checks run on them, how they are filtered and reshaped, how missing values are handled, and what is saved.

**Run this notebook first.**

| Stage | Input | Output |
|---|---|---|
| 1. Ingest | CDC PLACES API | `data/raw/places_county_2025.csv` + provenance record |
| 2–6. Inspect, validate, filter, convert, reshape | raw file | in-memory tables |
| 7. Missingness | wide table | complete-case table |
| 8. Save | complete-case table | `data/processed/county_analysis.csv` |

In [1]:
# Work from the repository root so the relative paths and `src` imports
# work whether the notebook is started from notebooks/ or from the root.
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Working directory:", ROOT)

import json
import numpy as np
import pandas as pd

from src.fetch import fetch_places, MEASURES, BASE_URL
from src.clean import (validate_raw, tidy_places, missingness_report,
                       drop_incomplete, EXCLUDE_STATES)

RAW_PATH = Path("data/raw/places_county_2025.csv")
PROVENANCE_PATH = Path("data/raw/places_county_2025_provenance.json")
PROCESSED_PATH = Path("data/processed/county_analysis.csv")

Y = "diabetes"
X_COLS = ["lpa", "obesity", "csmoking"]
ANALYSIS_COLS = [Y] + X_COLS

# A running log of how many rows survive each step
log = []
def record(step, df, note=""):
    log.append({"step": step, "rows": len(df), "note": note})

Working directory: /workspaces/STA-556-Final-Project


## 1. Data source and ingestion

- **Provider:** Centers for Disease Control and Prevention (CDC)
- **Dataset:** PLACES: Local Data for Better Health, County Data, 2025 release (Socrata ID `swc5-untb`)
- **Access:** Socrata Open Data API (SODA); public, no API key needed
- **What PLACES is:** model-based estimates of adult health outcomes and behaviors for every US county, built from the Behavioral Risk Factor Surveillance System (BRFSS) survey together with Census population data.

**Measures requested**

| Measure ID | Meaning | Role |
|---|---|---|
| `DIABETES` | Diagnosed diabetes among adults (%) | response |
| `LPA` | No leisure-time physical activity among adults (%) | main predictor |
| `OBESITY` | Obesity among adults (%) | control |
| `CSMOKING` | Current cigarette smoking among adults (%) | control |

We request **age-adjusted** prevalence so that counties with older or younger populations can be compared fairly.

`fetch_places` downloads the data only if `data/raw/places_county_2025.csv` does not exist yet. The API is paginated, so the function requests pages until the last one. The raw response is saved unchanged, and a provenance record (endpoint, query, retrieval time) is written next to it.

In [2]:
raw = fetch_places(MEASURES, RAW_PATH)
record("1. Raw API data", raw, "one row per county x measure")
print("Endpoint:", BASE_URL)
print("Raw file:", RAW_PATH, "| shape:", raw.shape)

Endpoint: https://data.cdc.gov/resource/swc5-untb.json
Raw file: data/raw/places_county_2025.csv | shape: (11832, 21)


In [3]:
# Provenance record written on the first download
if PROVENANCE_PATH.exists():
    print(json.dumps(json.loads(PROVENANCE_PATH.read_text()), indent=2))
else:
    print("No provenance file found (the raw file was added by hand).")

{
  "source": "CDC PLACES: Local Data for Better Health, County Data, 2025 release",
  "dataset_id": "swc5-untb",
  "endpoint": "https://data.cdc.gov/resource/swc5-untb.json",
  "query_parameters": {
    "$where": "measureid in('DIABETES','LPA','OBESITY','CSMOKING') AND data_value_type='Age-adjusted prevalence'",
    "$order": ":id"
  },
  "retrieved_utc": "2026-10-29T09:36:10+00:00",
  "n_rows": 11832,
  "file": "places_county_2025.csv"
}


## 2. Inspect the raw data

Before changing anything, look at the structure: the columns, their types, and how the rows are organized.

In [4]:
raw.head()

,year,stateabbr,statedesc,datasource,category,measure,data_value_unit,data_value_type,data_value,low_confidence_limit,...,totalpopulation,totalpop18plus,locationid,categoryid,measureid,datavaluetypeid,short_question_text,locationname,data_value_footnote_symbol,data_value_footnote
0,2023,US,United States,BRFSS,Health Risk Behaviors,Current cigarette smoking among adults,%,Age-adjusted prevalence,11.6,11.4,...,334914895,262083034,59,RISKBEH,CSMOKING,AgeAdjPrv,Current Cigarette Smoking,NaN,NaN,NaN
1,2023,US,United States,BRFSS,Health Outcomes,Diagnosed diabetes among adults,%,Age-adjusted prevalence,10.3,10.1,...,334914895,262083034,59,HLTHOUT,DIABETES,AgeAdjPrv,Diabetes,NaN,NaN,NaN
2,2023,US,United States,BRFSS,Health Risk Behaviors,No leisure-time physical activity among adults,%,Age-adjusted prevalence,23.7,23.4,...,334914895,262083034,59,RISKBEH,LPA,AgeAdjPrv,Physical Inactivity,NaN,NaN,NaN
3,2023,US,United States,BRFSS,Health Outcomes,Obesity among adults,%,Age-adjusted prevalence,32.9,32.5,...,334914895,262083034,59,HLTHOUT,OBESITY,AgeAdjPrv,Obesity,NaN,NaN,NaN
4,2023,AL,Alabama,BRFSS,Health Risk Behaviors,Current cigarette smoking among adults,%,Age-adjusted prevalence,14.5,11.6,...,60342,46253,01001,RISKBEH,CSMOKING,AgeAdjPrv,Current Cigarette Smoking,Autauga,NaN,NaN


In [5]:
# Column names and the types pandas assigned when reading the CSV
raw.dtypes.to_frame("dtype")

,dtype
year,int64
stateabbr,str
statedesc,str
datasource,str
category,str
measure,str
data_value_unit,str
data_value_type,str
data_value,float64
low_confidence_limit,float64


In [6]:
# The raw data are in LONG format: one row per county per measure
print("Rows per measure:")
print(raw["measureid"].value_counts(), "\n")
print("Value types present:", raw["data_value_type"].unique())
print("Distinct counties (locationid):", raw["locationid"].nunique())
print("Distinct state codes:", raw["stateabbr"].nunique())

Rows per measure:
measureid
CSMOKING    2958
DIABETES    2958
LPA         2958
OBESITY     2958
Name: count, dtype: int64 

Value types present: <StringArray>
['Age-adjusted prevalence']
Length: 1, dtype: str
Distinct counties (locationid): 2958
Distinct state codes: 50


*TODO: describe what you see, e.g. the data are long (county × measure), `data_value` is the prevalence, and each county should appear once per measure.*

## 3. Validate the raw data

`validate_raw` stops the workflow with a clear error if any of these fail:

1. all required columns are present;
2. all four measures were returned;
3. no county appears twice for the same measure;
4. every prevalence lies between 0 and 100.

It returns a few facts for the report.

In [7]:
facts = validate_raw(raw, MEASURES)
facts

{'n_rows': 11832, 'n_counties': 2958, 'n_states': 50, 'n_nonnumeric_values': 4}

## 4. Filter to the study population

PLACES covers the 50 states and DC. Any rows coded as the national total or a territory are removed so the unit of observation is a county in the 50 states + DC.

In [8]:
territory_rows = raw[raw["stateabbr"].isin(EXCLUDE_STATES)]
print("State codes excluded:", sorted(EXCLUDE_STATES))
print("Rows with an excluded code:", len(territory_rows))
territory_rows["stateabbr"].value_counts()

State codes excluded: ['AS', 'GU', 'MP', 'PR', 'US', 'VI']
Rows with an excluded code: 4


stateabbr
US    4
Name: count, dtype: int64

In [9]:
filtered = raw[raw["measureid"].isin(MEASURES) & ~raw["stateabbr"].isin(EXCLUDE_STATES)]
record("4. Keep 50 states + DC, four measures", filtered,
       f"removed {len(raw) - len(filtered)} rows")
print("States + DC remaining:", filtered["stateabbr"].nunique())

States + DC remaining: 49


## 5. Type conversion

The API sends every value as text. `data_value` (the prevalence) and `totalpopulation` must be numeric. Values that cannot be converted become missing (`NaN`) rather than raising an error, and we count them.

In [10]:
as_number = pd.to_numeric(filtered["data_value"], errors="coerce")
print("Non-numeric prevalence values (become NaN):", int(as_number.isna().sum()))
print(as_number.describe().round(2))

Non-numeric prevalence values (become NaN): 4
count    11824.00
mean        23.04
std         11.00
min          6.00
25%         13.20
50%         20.70
75%         32.82
max         54.00
Name: data_value, dtype: float64


## 6. Reshape: long to wide

The analysis needs **one row per county** with one column per measure. `tidy_places` pivots the long table:

```
locationid  measureid  data_value          locationid  diabetes  lpa  obesity  csmoking
01001       DIABETES   11.2         →      01001       11.2      27.1 36.0     17.9
01001       LPA        27.1
...
```

It also converts the types from step 5, keeps county identifiers (`locationid` = county FIPS code, state, county name), attaches county population, and renames the measures to lowercase.

In [11]:
wide = tidy_places(raw, MEASURES)
record("6. Reshape long → wide", wide, "one row per county")
print("Shape:", wide.shape)
wide.head()

Shape: (2957, 8)


,locationid,stateabbr,locationname,diabetes,lpa,obesity,csmoking,totalpopulation
0,01001,AL,Autauga,11.4,26.0,39.5,14.5,60342
1,01003,AL,Baldwin,10.4,23.7,35.0,13.3,253507
2,01005,AL,Barbour,16.5,34.5,47.1,20.8,24585
3,01007,AL,Bibb,13.9,31.9,43.7,19.9,21868
4,01009,AL,Blount,11.3,27.7,36.0,16.9,59816


In [12]:
# Checks on the reshape
assert wide["locationid"].is_unique, "a county appears more than once"
expected = filtered["locationid"].nunique()
print(f"Counties after reshape: {len(wide)} (expected {expected})")
assert len(wide) == expected

Counties after reshape: 2957 (expected 2957)


## 7. Missingness

A county may lack an estimate for one or more measures. We report how much is missing, where it is, and then decide how to handle it.

In [13]:
missingness_report(wide, ANALYSIS_COLS)

,n_missing,pct_missing
diabetes,1,0.03
lpa,1,0.03
obesity,1,0.03
csmoking,1,0.03


In [14]:
# Where are the incomplete counties?
incomplete = wide[wide[ANALYSIS_COLS].isna().any(axis=1)]
print("Counties missing at least one analysis variable:", len(incomplete))
incomplete[["stateabbr", "locationname"] + ANALYSIS_COLS].head(20)

Counties missing at least one analysis variable: 1


,stateabbr,locationname,diabetes,lpa,obesity,csmoking
2488,TX,Loving,NaN,NaN,NaN,NaN


In [15]:
county, n_dropped = drop_incomplete(wide, ANALYSIS_COLS)
record("7. Complete cases", county, f"dropped {n_dropped} incomplete counties")
print(f"Dropped {n_dropped} counties; {len(county)} remain "
      f"({100 * n_dropped / len(wide):.2f}% of counties).")

Dropped 1 counties; 2956 remain (0.03% of counties).


*TODO: justify the decision. If only a handful of counties are dropped and they are not concentrated in one region, complete-case analysis is unlikely to bias the results. Say how many were dropped and from where.*

## 8. Final checks and save

Confirm the analysis table is what the analysis notebook expects, then save it. The processed file is regenerated by this notebook and is never edited by hand.

In [16]:
assert county[ANALYSIS_COLS].notna().all().all()
assert county[ANALYSIS_COLS].apply(lambda s: s.between(0, 100)).all().all()
assert county["locationid"].is_unique

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
county.to_csv(PROCESSED_PATH, index=False)
print("Saved:", PROCESSED_PATH, "| shape:", county.shape)
county.describe().round(2)

Saved: data/processed/county_analysis.csv | shape: (2956, 8)


,diabetes,lpa,obesity,csmoking,totalpopulation
count,2956.00,2956.00,2956.00,2956.00,2956.00
mean,11.17,27.02,37.63,16.34,107383.97
std,2.33,5.39,4.74,3.54,339570.85
min,6.00,12.20,17.10,6.50,81.00
25%,9.50,23.40,35.10,14.10,10486.50
50%,10.70,26.50,38.20,16.10,25799.50
75%,12.50,30.50,40.60,18.40,69272.50
max,23.80,48.90,54.00,40.30,9663345.00


## 9. Data dictionary for the processed file

`data/processed/county_analysis.csv` has one row per county:

In [17]:
dictionary = pd.DataFrame({
    "column": ["locationid", "stateabbr", "locationname", "diabetes", "lpa",
               "obesity", "csmoking", "totalpopulation"],
    "description": ["County FIPS code", "State abbreviation", "County name",
                    "Diagnosed diabetes, age-adjusted % of adults",
                    "No leisure-time physical activity, age-adjusted % of adults",
                    "Obesity, age-adjusted % of adults",
                    "Current smoking, age-adjusted % of adults",
                    "County population"],
})
dictionary

,column,description
0,locationid,County FIPS code
1,stateabbr,State abbreviation
2,locationname,County name
3,diabetes,"Diagnosed diabetes, age-adjusted % of adults"
4,lpa,"No leisure-time physical activity, age-adjuste..."
5,obesity,"Obesity, age-adjusted % of adults"
6,csmoking,"Current smoking, age-adjusted % of adults"
7,totalpopulation,County population


## 10. Processing log

Rows remaining after each step:

In [18]:
pd.DataFrame(log)

,step,rows,note
0,1. Raw API data,11832,one row per county x measure
1,"4. Keep 50 states + DC, four measures",11828,removed 4 rows
2,6. Reshape long → wide,2957,one row per county
3,7. Complete cases,2956,dropped 1 incomplete counties


**Summary.** *TODO: one short paragraph: raw rows → counties analysed, what was removed and why.*

Next: open **`notebooks/02_analysis.ipynb`**.